# Étape 1 — Ingestion des documents

**Objectif de ce notebook :**
- Charger les documents présents dans `data/docs/` (.txt, .md, .pdf)
- Les découper en chunks (morceaux de texte)
- Explorer et vérifier visuellement le résultat avant de passer à l'étape 2 (embeddings + FAISS)

Ce notebook réutilise les fonctions de `src/ingest.py` (pour éviter de dupliquer le code
entre l'exploration en notebook et le pipeline final en scripts), mais te permet
d'inspecter chaque résultat intermédiaire pas à pas.

**Kernel à utiliser :** `Python (RAG GED)` (créé par `setup_env.sh`).

## 0. Imports et accès au dossier `src/`

In [1]:
import sys
from pathlib import Path

# Permet d'importer les modules du dossier src/ depuis ce notebook
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

print("Racine du projet :", PROJECT_ROOT)
print("Dossier src ajouté au path :", SRC_DIR)

Racine du projet : c:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged
Dossier src ajouté au path : c:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\src


In [2]:
from ingest import load_documents, split_documents, DATA_DIR, CHUNK_SIZE, CHUNK_OVERLAP

print("Dossier des documents :", DATA_DIR)
print("Taille de chunk par défaut :", CHUNK_SIZE)
print("Chevauchement par défaut :", CHUNK_OVERLAP)

Dossier des documents : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs
Taille de chunk par défaut : 800
Chevauchement par défaut : 150


## 1a. Chargement des documents

On charge tous les fichiers `.txt`, `.md` et `.pdf` présents dans `data/docs/`.
Chaque fichier devient un ou plusieurs objets `Document` (texte brut + métadonnées,
dont le chemin du fichier source).

In [3]:
documents = load_documents(DATA_DIR)

print(f"\n{len(documents)} document(s) chargé(s).\n")
for doc in documents:
    print("Source :", doc.metadata.get("source"))
    print("Longueur du texte :", len(doc.page_content), "caractères")
    print("---")

[ingest] Chargé : exemple_atelier_preparation.md
[ingest] Chargé : GDS_GED_Services (4).pdf

23 document(s) chargé(s).

Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\exemple_atelier_preparation.md
Longueur du texte : 2375 caractères
---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
Longueur du texte : 276 caractères
---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
Longueur du texte : 2461 caractères
---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
Longueur du texte : 1502 caractères
---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
Longueur du texte : 263 caractères
---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\GDS_GED_Services (4).pdf
Longueur du texte : 183 caractères
---
Source : C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\do

In [4]:
# Aperçu du contenu brut du premier document
print(documents[0].page_content[:800])

# Processus métier — GDS GED Services (document d'exemple pour test)

## Atelier Réception

Les dossiers papier arrivent chez GDS GED Services et sont regroupés en lots.
Chaque lot reçoit un code-barres unique qui permettra de suivre le lot tout au long
du processus de numérisation, jusqu'à l'archivage.

## Atelier Préparation

L'atelier Préparation consiste à :
1. Ouvrir chaque dossier du lot.
2. Classer les pièces à l'intérieur du dossier dans le bon ordre.
3. Vérifier la complétude du dossier (toutes les pièces attendues sont présentes).
4. Retirer les agrafes et trombones qui pourraient endommager le scanner.

Une erreur fréquente à ce stade est le classement accidentel d'une page dans le
mauvais dossier. Cette erreur peut ensuite passer inaperçue lors du contrôle qualité
manuel si ell


## 1b. Découpage en chunks (chunking)

On découpe chaque document en chunks de `CHUNK_SIZE` caractères, avec un chevauchement
de `CHUNK_OVERLAP` caractères entre deux chunks consécutifs (pour ne pas couper une
information importante pile à la frontière).

Le découpage respecte en priorité les titres Markdown et les paragraphes avant de
couper au milieu d'une phrase si nécessaire (voir `separators` dans `ingest.py`).

In [5]:
chunks = split_documents(documents)

print(f"{len(chunks)} chunks générés à partir de {len(documents)} document(s).")

lengths = [len(c.page_content) for c in chunks]
print("Longueur moyenne des chunks :", sum(lengths) / len(lengths))
print("Longueur min / max :", min(lengths), "/", max(lengths))

63 chunks générés à partir de 23 document(s).
Longueur moyenne des chunks : 635.2222222222222
Longueur min / max : 173 / 799


In [6]:
# Affichage des 3 premiers chunks avec leur source
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} (source: {chunk.metadata.get('source')}) ---")
    print(chunk.page_content)
    print()

--- Chunk 0 (source: C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\exemple_atelier_preparation.md) ---
# Processus métier — GDS GED Services (document d'exemple pour test)

## Atelier Réception

Les dossiers papier arrivent chez GDS GED Services et sont regroupés en lots.
Chaque lot reçoit un code-barres unique qui permettra de suivre le lot tout au long
du processus de numérisation, jusqu'à l'archivage.

--- Chunk 1 (source: C:\Users\DELL\Desktop\PROJET_PFA\rag-assistant-ged\data\docs\exemple_atelier_preparation.md) ---
## Atelier Préparation

L'atelier Préparation consiste à :
1. Ouvrir chaque dossier du lot.
2. Classer les pièces à l'intérieur du dossier dans le bon ordre.
3. Vérifier la complétude du dossier (toutes les pièces attendues sont présentes).
4. Retirer les agrafes et trombones qui pourraient endommager le scanner.

Une erreur fréquente à ce stade est le classement accidentel d'une page dans le
mauvais dossier. Cette erreur peut ensuite passer inaperçue lo

## 1c. Expérimenter avec d'autres paramètres de chunking (optionnel)

Si tu trouves que les chunks sont trop gros (le LLM reçoit trop de texte non pertinent)
ou trop petits (le contexte est incomplet), tu peux tester d'autres valeurs ici avant
de les reporter dans `src/ingest.py`.

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Essaie de modifier ces valeurs et relance la cellule pour comparer
TEST_CHUNK_SIZE = 500
TEST_CHUNK_OVERLAP = 100

test_splitter = RecursiveCharacterTextSplitter(
    chunk_size=TEST_CHUNK_SIZE,
    chunk_overlap=TEST_CHUNK_OVERLAP,
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " ", ""],
)
test_chunks = test_splitter.split_documents(documents)

print(f"Avec chunk_size={TEST_CHUNK_SIZE}, overlap={TEST_CHUNK_OVERLAP} : {len(test_chunks)} chunks")
print("\n--- Exemple ---")
print(test_chunks[0].page_content)

Avec chunk_size=500, overlap=100 : 103 chunks

--- Exemple ---
# Processus métier — GDS GED Services (document d'exemple pour test)

## Atelier Réception

Les dossiers papier arrivent chez GDS GED Services et sont regroupés en lots.
Chaque lot reçoit un code-barres unique qui permettra de suivre le lot tout au long
du processus de numérisation, jusqu'à l'archivage.


## Conclusion et prochaine étape

Une fois que la découpe te semble cohérente (chunks ni trop longs ni trop courts,
information pas coupée n'importe où), tu peux :

1. Ajuster `CHUNK_SIZE` / `CHUNK_OVERLAP` dans `src/ingest.py` si besoin, avec les
   valeurs que tu as testées ci-dessus.
2. Passer au notebook **`02_embeddings_index.ipynb`** (Étape 2 : embeddings + index FAISS).